# Fase 4 — índice FAISS de mensagens com fonte

Este notebook prepara um índice a partir da conversa **fictícia**, usando vetores de fixture. A busca carrega o índice já salvo; não reindexa a cada pergunta. Não há chamada de API nem interpretação semântica por LLM nesta demonstração.


In [ ]:
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0,str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.indice_fontes import IndiceFontes


In [ ]:
class EmbeddingsFixture:
    def vetor(self,texto):
        texto = texto.lower()
        return [1.0 if 'alfa' in texto else 0.0, 1.0 if 'beta' in texto else 0.0, 0.1]
    def embed_documents(self,textos):
        return [self.vetor(texto) for texto in textos]
    def embed_query(self,texto):
        return self.vetor(texto)


## Preparar, carregar e conferir a fonte
A resposta recuperada contém `mensagem_id` e o texto original do SQLite. Ela é evidência de onde algo foi dito; não comprova pagamento ou saldo.


In [ ]:
with tempfile.TemporaryDirectory() as pasta:
    with Banco(Path(pasta)/'demo.sqlite3') as banco:
        banco.preparar_demo()
        ana, bruno = Contexto('demo','ana'), Contexto('demo','bruno')
        corpus = banco.selecionar_corpus(ana,'Demonstração fictícia')
        banco.importar(ana,corpus,raiz/'dados/ficticios/conversa_empresa.txt')
        indice = IndiceFontes(banco,Path(pasta)/'indices')
        fixture = EmbeddingsFixture()
        preparo = indice.preparar(ana,corpus,fixture,'fixture_manual_v1')
        assert preparo['estado'] == 'preparado'
        assert indice.preparar(ana,corpus,fixture,'fixture_manual_v1')['estado'] == 'atual'
        achados = indice.buscar(ana,corpus,'Alfa',fixture,'fixture_manual_v1',limite=2)
        assert 'Alfa' in achados[0]['texto']
        print('Versão:',preparo['versao_corpus'][:12], 'Fonte:',achados[0]['mensagem_id'])
        try:
            indice.buscar(bruno,corpus,'Alfa',fixture,'fixture_manual_v1')
        except PermissionError:
            print('Corpus privado: acesso de Bruno bloqueado')
        else:
            raise AssertionError('Falha de escopo')


## Uso no aplicativo
O menu 11 prepara o corpus selecionado com embeddings reais apenas após confirmação. No menu 9, a ferramenta `buscar_mensagens_fonte` carrega o índice existente e retorna fontes autorizadas. O notebook verifica a infraestrutura com fixture; a trajetória real do agente ainda exige teste manual.
